In [60]:
import sys, os
sys.path.append(os.path.abspath(os.path.join('..')))

In [61]:
import numpy as np
import pandas as pd
import scipy.stats as stats
import math

In [62]:
from src.data import load_csv

In [63]:
df = load_csv("../data/hour.csv")

In [64]:
X = df.iloc[:, :13]
Y = df.iloc[:, -1]

In [65]:
X

,dteday,season,yr,mnth,hr,holiday,weekday,workingday,weathersit,temp,atemp,hum,windspeed
0,2011-01-01,1,0,1,0,0,6,0,1,0.24,0.2879,0.81,0.0000
1,2011-01-01,1,0,1,1,0,6,0,1,0.22,0.2727,0.80,0.0000
2,2011-01-01,1,0,1,2,0,6,0,1,0.22,0.2727,0.80,0.0000
3,2011-01-01,1,0,1,3,0,6,0,1,0.24,0.2879,0.75,0.0000
4,2011-01-01,1,0,1,4,0,6,0,1,0.24,0.2879,0.75,0.0000
...,...,...,...,...,...,...,...,...,...,...,...,...,...
17374,2012-12-31,1,1,12,19,0,1,1,2,0.26,0.2576,0.60,0.1642
17375,2012-12-31,1,1,12,20,0,1,1,2,0.26,0.2576,0.60,0.1642
17376,2012-12-31,1,1,12,21,0,1,1,1,0.26,0.2576,0.60,0.1642
17377,2012-12-31,1,1,12,22,0,1,1,1,0.26,0.2727,0.56,0.1343


In [66]:
Y

0         16
1         40
2         32
3         13
4          1
        ... 
17374    119
17375     89
17376     90
17377     61
17378     49
Name: cnt, Length: 17379, dtype: int64

In [67]:
from sklearn.model_selection import train_test_split
from src.features import split_sets_tr_val
from src.features import clean_and_preprocess_data
from src.models import evaluate_variant, MultipleLinearRegression

X_train_full, X_test_full, y_train_full, y_test_full = train_test_split(X, Y, test_size=0.2, shuffle=False)

features = ["holiday", "yr", "season", "weathersit", "hr", "atemp", "hum", "windspeed"]
X_tr, y_tr, X_val, y_val = split_sets_tr_val(X_train_full, y_train_full, feature_cols=features)

In [68]:
y_val

11122    352
11123    463
11124    738
11125    671
11126    427
        ... 
13898    502
13899    705
13900    327
13901    250
13902    214
Name: cnt, Length: 2781, dtype: int64

In [69]:
X_tr, y_tr_log, X_val, y_val_log = clean_and_preprocess_data(df, target_cols='cnt', feature_cols=features)

In [70]:
X_tr

,holiday,yr,atemp,hum,windspeed,season_2,season_3,season_4,weathersit_2,weathersit_3,...,hr_14,hr_15,hr_16,hr_17,hr_18,hr_19,hr_20,hr_21,hr_22,hr_23
0,0,0,0.2879,0.81,0.0000,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
1,0,0,0.2727,0.80,0.0000,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2,0,0,0.2727,0.80,0.0000,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
3,0,0,0.2879,0.75,0.0000,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
4,0,0,0.2879,0.75,0.0000,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
11117,0,1,0.4394,0.38,0.1940,1.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
11118,0,1,0.4545,0.36,0.1343,1.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
11119,0,1,0.4848,0.31,0.0000,1.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
11120,0,1,0.5000,0.27,0.2836,1.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


Outliers and capping them !! it will be fun !!

Leverage hii

In [71]:
def compute_leverage(X):
    X = np.asarray(X, dtype=float)

    X = np.c_[np.ones(X.shape[0]), X]    

    XTX_inv = np.linalg.pinv(X.T @ X)

    X_projected = X @ XTX_inv


    hadamard_grid = X_projected * X

    h = np.sum(hadamard_grid, axis=1)

    return h

In [72]:
X_tr = np.c_[np.ones(X_tr.shape[0]), X_tr] 
p = X_tr.shape[1]
n = X_tr.shape[0]

print(n, p)

11122 34


In [73]:
leverage_mat = compute_leverage(X_tr)

print(leverage_mat)

print(leverage_mat.shape)

print(np.isclose(np.sum(leverage_mat), p))

print(np.isclose(np.mean(leverage_mat), p/n))


threshold = 2 * np.mean(leverage_mat) 
print(threshold)

print(len(np.where(leverage_mat > threshold)[0]))

[0.00284137 0.00281202 0.00285036 ... 0.0035213  0.00302401 0.00357863]
(11122,)
True
True
0.006114008271893542
109


for test:

In [74]:
leverage_mat = compute_leverage(X_val)

print(leverage_mat)

print(leverage_mat.shape)

print(np.isclose(np.sum(leverage_mat), p))

print(np.isclose(np.mean(leverage_mat), p/n))


threshold = 2 * np.mean(leverage_mat) 
print(threshold)

print(len(np.where(leverage_mat > threshold)[0]))

[0.01055259 0.01041973 0.01088811 ... 0.01023761 0.0105848  0.0107253 ]
(2781,)
False
False
0.022294138798993134
72


In [75]:
def studentised_residual(y_true, y_pred, h, p, n):
    y_true = np.asarray(y_true, float).ravel()
    y_pred = np.asarray(y_pred, float).ravel()
    h = np.asarray(h, float).ravel()
    
    e = y_true - y_pred

    RSS = np.sum(e**2)
    sigma_hat = np.sqrt(RSS / (n-p))

    # scale raw errors by leverage adjusted standard deviation
    one_minus_h = np.maximum(1e-12, 1.0 - h)
    se_residual = sigma_hat * np.sqrt(one_minus_h)

    r = np.divide(e, se_residual)

    return r


In [76]:
X_tr, y_tr_log, X_val, y_val_log = clean_and_preprocess_data(df, target_cols='cnt', feature_cols=features)
leverage_mat_test = compute_leverage(X_val)


metrics, summary, y_pred, y_val_true = evaluate_variant(X_tr, y_tr_log, X_val, y_val_log)
print(y_pred.shape, y_val_true.shape)

from IPython.display import display
display(metrics)

r = studentised_residual(y_val_true, y_pred, leverage_mat_test, p, n)
print(r)

print(np.sum(np.abs(r) > 3.0))



(2781, 1) (2781, 1)


,MSE,MAE,RMSE,Val R2,Train R2,Adjusted_R2_Val,SSE,SST,SSM
0,0.361793,0.460404,0.601492,0.806653,0.814733,0.804331,1006.14683,5203.849792,4439.528489


[ 0.14389221  0.11248701  0.2302409  ... -0.89091336 -0.8688457
 -1.97218668]
366


In [77]:
def cooks_distance(r, p, h,):
    r = np.asarray(r, float)
    h = np.asarray(h, float)

    r_squared = r ** 2
    target_term = r_squared / p

    leverage_term = h / (1 - h)

    D = target_term * leverage_term

    return D



# for test
D = cooks_distance(r, p, leverage_mat_test)
print(D)
n = X_val.shape[0]

threshold = 4/ n

print(np.sum(D > threshold))

print("Max Cook's Distance: ", np.max(D))

[6.49474298e-06 3.91860452e-06 1.71630018e-05 ... 2.41468017e-04
 2.37525603e-04 1.24025140e-03]
702
Max Cook's Distance:  0.02779955174849517


In [78]:
# <!-- for our dear train dataset, let us calculate r and D -->
n = X_tr.shape[0]
p = X_tr.shape[1]

model = MultipleLinearRegression()
model.fit(X_tr, y_tr_log)
y_pred = model.predict(X_tr)

leverage_mat_train = compute_leverage(X_tr)
leverage_threshold = 2 * np.mean(leverage_mat_train) 

print("Rows above leverage threshold: ", np.sum(leverage_mat_train > leverage_threshold))

r = studentised_residual(y_pred, y_tr_log, leverage_mat_train, p, n)
print("Target Residuals above the limit: ", np.sum(np.abs(r) > 3))

D = cooks_distance(r, p, leverage_mat_train)

print(D)
threshold = 4/ n

print("Rows having Cook's distance greater than threshold:", np.sum(D > threshold))

print("Max Cook's Distance: ", np.max(D))


Rows above leverage threshold:  109
Target Residuals above the limit:  100
[2.59020017e-06 4.45878464e-04 6.14697971e-04 ... 2.25619702e-06
 7.68668289e-07 9.27572517e-07]
Rows having Cook's distance greater than threshold: 687
Max Cook's Distance:  0.0042280122302388225


In [79]:
def cooks_pruning(X_tr, y_tr, D, X_val, y_val):
    cutoff_99 = np.percentile(D, 99)

    keep_mask = D <= cutoff_99

    X_tr_pruned = X_tr[keep_mask]
    y_tr_pruned = y_tr[keep_mask]

    print(f"Pruned {np.sum(~keep_mask)} rows out of {X_tr.shape[0]}")

    model_full = MultipleLinearRegression()
    model_full.fit(X_tr, y_tr)

    model_pruned = MultipleLinearRegression()
    model_pruned.fit(X_tr_pruned, y_tr_pruned)

    delta_beta = model_full.beta_hat - model_pruned.beta_hat
    delta_beta_norm = np.linalg.norm(delta_beta)

    print(f"Euclidean Weight Shift: {delta_beta_norm:.6f}")

    r2_before = model_full.r2_score(X_val, y_val)
    r2_after = model_pruned.r2_score(X_val, y_val)

    print(f"Validation R2: Full train:{ r2_before:.6f}")
    print(f"Validation R2: Pruned data: {r2_after:.6f}")


cooks_pruning(X_tr, y_tr_log, D, X_val, y_val_log)


Pruned 112 rows out of 11122
Euclidean Weight Shift: 0.216371
Validation R2: Full train:0.806653
Validation R2: Pruned data: 0.808784


In [80]:
def outlier_stress_test(X_tr, y_tr, leverage_mat_train,p, n):
    leverage_mat_train = np.asarray(leverage_mat_train, float).ravel()
    idx_max_h = np.argmax(leverage_mat_train)
    h_val = leverage_mat_train[idx_max_h]

    #injecting a massive spike:
    y_corrupted = np.asarray(y_tr, float).copy().ravel()
    spike = 10 * float(np.asarray(y_tr, float).std())
    y_corrupted[idx_max_h] = y_corrupted[idx_max_h] + spike

    model_corrupted = MultipleLinearRegression()
    model_corrupted.fit(X_tr, y_corrupted)
    y_pred_corrupted = model_corrupted.predict(X_tr)

    r_corrupted = studentised_residual(y_corrupted, y_pred_corrupted, leverage_mat_train, p, n)
    D_corrupted = cooks_distance(r_corrupted, p, leverage_mat_train)

    print("Injected row index", idx_max_h)
    print("Leverage h_ii: ", h_val)
    print("Corrupted Residual: (r_i): ", r_corrupted[idx_max_h])
    print("Corrupted Cook's Distance: ", D_corrupted[idx_max_h])

outlier_stress_test(X_tr, y_tr_log, leverage_mat_train, p, n)

Injected row index 5843
Leverage h_ii:  0.0074566433069997145
Corrupted Residual: (r_i):  22.226415465755217
Corrupted Cook's Distance:  0.11246536532079289


In [81]:
def influence():
    pass

In [82]:
def influential_diagnositcs():
    """here we call all three or four above functions and return a table of diagnositcs report! """

r2 from mlr vs r2 from src.metrics

In [84]:
features = ["holiday", "yr", "season", "weathersit", "hr", "atemp", "hum", "windspeed"]
print(y_train_full)
X_tr, y_tr_log, X_val, y_val_log = clean_and_preprocess_data(df, target_cols='cnt', feature_cols=features)
display(X_tr)
leverage_mat_test = compute_leverage(X_val)


metrics, summary, y_pred, y_val_true = evaluate_variant(X_tr, y_tr_log, X_val, y_val_log)
print(y_pred.shape, y_val_true.shape)
mlr =MultipleLinearRegression()
mlr.fit(X_tr, y_tr_log)
print("MLR module R2:")
print(mlr.r2_score(X_val, y_val))

metrics, summary, y_pred, y_val = evaluate_variant(X_tr, y_tr_log, X_val, y_val_log)
display(metrics)
display(summary)

0         16
1         40
2         32
3         13
4          1
        ... 
13898    502
13899    705
13900    327
13901    250
13902    214
Name: cnt, Length: 13903, dtype: int64


,holiday,yr,atemp,hum,windspeed,season_2,season_3,season_4,weathersit_2,weathersit_3,...,hr_14,hr_15,hr_16,hr_17,hr_18,hr_19,hr_20,hr_21,hr_22,hr_23
0,0,0,0.2879,0.81,0.0000,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
1,0,0,0.2727,0.80,0.0000,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2,0,0,0.2727,0.80,0.0000,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
3,0,0,0.2879,0.75,0.0000,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
4,0,0,0.2879,0.75,0.0000,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
11117,0,1,0.4394,0.38,0.1940,1.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
11118,0,1,0.4545,0.36,0.1343,1.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
11119,0,1,0.4848,0.31,0.0000,1.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
11120,0,1,0.5000,0.27,0.2836,1.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


(2781, 1) (2781, 1)
MLR module R2:
0.8066533682249675


,MSE,MAE,RMSE,Val R2,Train R2,Adjusted_R2_Val,SSE,SST,SSM
0,0.361793,0.460404,0.601492,0.806653,0.814733,0.804331,1006.14683,5203.849792,4439.528489


,Beta Hat,Standard Error,t_stats,p_values,CI lower,CI upper
Intercept,2.593241,0.042471,61.059413,0.000000e+00,2.509991,2.676491
holiday,-0.141370,0.034741,-4.069282,4.748509e-05,-0.209469,-0.073272
yr,0.598742,0.015883,37.695895,0.000000e+00,0.567608,0.629877
atemp,1.775040,0.053618,33.105275,0.000000e+00,1.669940,1.880141
hum,-0.206435,0.038001,-5.432314,5.680342e-08,-0.280924,-0.131946
windspeed,-0.192609,0.048745,-3.951355,7.819177e-05,-0.288157,-0.097060
season_2,0.364013,0.018875,19.285350,0.000000e+00,0.327015,0.401012
season_3,0.342931,0.026087,13.145905,0.000000e+00,0.291797,0.394065
season_4,0.549819,0.019294,28.497564,0.000000e+00,0.512001,0.587638
weathersit_2,-0.045624,0.014308,-3.188737,1.432932e-03,-0.073670,-0.017578
